# E3 — Entrenamiento PCN v2

Continúa el entrenamiento desde `best.pt` (época 97) con mejoras:
- **300 épocas totales** (~200 épocas más)
- **`--w_coarse 1.0`**: mayor peso en la pérdida coarse → el modelo aprende estructura geométrica antes de refinar
- **LR cálido**: warm restart en 5e-5, un solo decay a la mitad en época ~200
- **Batch 64**: la T4 aguanta el doble de batch

⏱️ **Tiempo estimado en T4: ~2 horas**

---
### Antes de ejecutar:
Menú → **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELDA 2: Instalar dependencias y crear estructura de carpetas ──
import os
from pathlib import Path

!pip install torch numpy matplotlib --quiet

REPO_DIR = '/content/TFM'
for d in ['E3', 'Datos/sintetico', 'Datos/fantastic_breaks', 'E3/checkpoints']:
    Path(f'{REPO_DIR}/{d}').mkdir(parents=True, exist_ok=True)

# Necesario para que Python trate E3 como modulo
Path(f'{REPO_DIR}/E3/__init__.py').touch()

os.chdir(REPO_DIR)
print('Listo. Directorio:', os.getcwd())

In [ ]:
# ── CELDA 3: RUTAS DE DRIVE ────────────────────────────────────

DRIVE = '/content/drive/MyDrive'

# Checkpoint del primer entrenamiento (best.pt epoca 97)
RUTA_BEST_PT = f'{DRIVE}/Datos_E2_E3/E3/checkpoints_pcn/best.pt'

# Datos de entrenamiento
RUTA_SINTETICO    = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas'            # 2367 pares sinteticos
RUTA_FB_PROCESADO = f'{DRIVE}/Datos_E2_E3/General/Fantastik_Break_Preprocesado' # 61 pares reales

# Carpeta donde guardar el checkpoint v2 al final
RUTA_SALIDA_DRIVE = f'{DRIVE}/Datos_E2_E3/E3/checkpoints_pcn'

print('Rutas configuradas:')
print(f'  best.pt         : {RUTA_BEST_PT}')
print(f'  sintetico       : {RUTA_SINTETICO}')
print(f'  fantastic_breaks: {RUTA_FB_PROCESADO}')
print(f'  salida Drive    : {RUTA_SALIDA_DRIVE}')

In [ ]:
# ── CELDA 4: Copiar datos y código E3 desde Drive ──────────────
import shutil
from pathlib import Path

def copiar_si_falta(src, dst, es_dir=True):
    src, dst = Path(src), Path(dst)
    if dst.exists():
        n = len(list(dst.rglob('*'))) if dst.is_dir() else 1
        print(f'  [OK] ya existe: {dst}  ({n} archivos)')
        return
    if not src.exists():
        print(f'  [ERROR] no encontrado en Drive: {src}')
        return
    print(f'  Copiando {src.name} ...', end=' ')
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst) if es_dir else shutil.copy2(src, dst)
    print('listo.')

# Datos de entrenamiento
copiar_si_falta(RUTA_SINTETICO,    'Datos/sintetico/roturas',          es_dir=True)
copiar_si_falta(RUTA_FB_PROCESADO, 'Datos/fantastic_breaks/procesado', es_dir=True)

# Checkpoint del primer entrenamiento
copiar_si_falta(RUTA_BEST_PT, 'E3/checkpoints/best.pt', es_dir=False)

# Código E3 desde Raquel/E3/ en Drive
DRIVE = '/content/drive/MyDrive'
for f in ['dataset.py', 'train.py']:
    copiar_si_falta(f'{DRIVE}/Raquel/E3/{f}', f'E3/{f}', es_dir=False)

# Aplicar mejora w_coarse=1.0 en train.py (el Drive tiene 0.5 hardcodeado)
train_path = Path('E3/train.py')
codigo = train_path.read_text(encoding='utf-8')
if '+ 0.5 * chamfer_distance(coarse' in codigo:
    codigo = codigo.replace(
        '+ 0.5 * chamfer_distance(coarse, completo)',
        '+ 1.0 * chamfer_distance(coarse, completo)'
    )
    train_path.write_text(codigo, encoding='utf-8')
    print('  train.py parcheado: w_coarse 0.5 → 1.0')
else:
    print('  train.py ya tiene w_coarse actualizado')

# Resumen
print()
for carpeta in ['Datos/sintetico/roturas', 'Datos/fantastic_breaks/procesado']:
    n = len(list(Path(carpeta).glob('*.npy'))) if Path(carpeta).exists() else 0
    print(f'  {carpeta}: {n} archivos .npy')
print(f'  best.pt: {Path("E3/checkpoints/best.pt").exists()}')

In [ ]:
# ── CELDA 5: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU disponible: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('AVISO: no hay GPU. Ve a Entorno de ejecucion → Cambiar tipo → T4 GPU')

In [ ]:
# ── CELDA 6: ENTRENAR v2 ───────────────────────────────────────
# Deja esta celda corriendo (~2 horas en T4).
# w_coarse=1.0 ya está aplicado en train.py por la Celda 4.

!python -m E3.train \
    --resume     E3/checkpoints/best.pt \
    --epochs     300 \
    --lr         5e-5 \
    --lr_decay   100 \
    --batch_size 64

In [ ]:
# ── CELDA 7: Guardar en Drive ──────────────────────────────────
import shutil, time
from pathlib import Path

Path(RUTA_SALIDA_DRIVE).mkdir(parents=True, exist_ok=True)

ts = time.strftime('%Y%m%d_%H%M')
dst = f'{RUTA_SALIDA_DRIVE}/best_v2_{ts}.pt'
shutil.copy2('E3/checkpoints/best.pt', dst)
print(f'Checkpoint v2 guardado en Drive: {dst}')

In [ ]:
# ── CELDA 8: Evaluacion del modelo v2 ─────────────────────────
import torch, numpy as np
from pathlib import Path
from E3.dataset import construir_dataloaders
from E3.train import PCN, chamfer_distance

SALIDA = Path('E3/resultados_v2')
SALIDA.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

_, _, test_loader = construir_dataloaders(
    carpetas=['Datos/fantastic_breaks/procesado', 'Datos/sintetico/roturas'],
    batch_size=32, augmentar=False,
)

model = PCN().to(device)
ckpt = torch.load('E3/checkpoints/best.pt', map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Checkpoint epoca {ckpt["epoch"]} cargado. Test set: {len(test_loader.dataset)} muestras\n')

cds, muestras = [], []
with torch.no_grad():
    for roto, gt in test_loader:
        roto, gt = roto.to(device), gt.to(device)
        _, fine = model(roto)
        for j in range(roto.size(0)):
            cd = chamfer_distance(fine[j:j+1], gt[j:j+1]).item()
            cds.append(cd)
            muestras.append((roto[j].cpu().numpy(), gt[j].cpu().numpy(),
                             fine[j].cpu().numpy(), cd))

cds = np.array(cds)
print(f'Chamfer Distance (CD-L1):')
print(f'  media  : {cds.mean():.6f}  (v1 fue 0.076587)')
print(f'  mediana: {np.median(cds):.6f}')
print(f'  mejor  : {cds.min():.6f}  (muestra {cds.argmin()})')
print(f'  peor   : {cds.max():.6f}  (muestra {cds.argmax()})')

In [ ]:
# ── CELDA 9: Ver figuras directamente en Colab ─────────────────
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
from pathlib import Path

SALIDA = Path('E3/resultados_v2')

def guardar_figura(roto, gt, pred, titulo, ruta):
    fig = plt.figure(figsize=(20, 5))
    fig.suptitle(titulo, fontsize=10)
    for i, (pts, label, color) in enumerate([
        (roto, 'Entrada (rota)',  '#4C72B0'),
        (gt,   'GT (completa)',   '#55A868'),
        (pred, 'Prediccion',      '#C44E52'),
    ], 1):
        ax = fig.add_subplot(1, 4, i, projection='3d')
        ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=color, s=2, alpha=0.7)
        ax.set_title(label, fontsize=9); ax.set_axis_off()
        ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-1,1)
        ax.view_init(elev=20, azim=45)
    ax4 = fig.add_subplot(1, 4, 4, projection='3d')
    ax4.scatter(gt[:,0],   gt[:,1],   gt[:,2],   c='#AAAAAA', s=2, alpha=0.25)
    ax4.scatter(pred[:,0], pred[:,1], pred[:,2], c='#DD8452', s=2, alpha=0.8)
    ax4.set_title('Overlay GT+Pred', fontsize=9); ax4.set_axis_off()
    ax4.set_xlim(-1,1); ax4.set_ylim(-1,1); ax4.set_zlim(-1,1)
    ax4.view_init(elev=20, azim=45)
    plt.tight_layout()
    plt.savefig(ruta, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

orden = np.argsort(cds)
indices = list(orden[:4]) + list(orden[-4:])
etiquetas = ['mejor']*4 + ['peor']*4

print('=== MEJORES 4 ===')
for rank, (idx, etiq) in enumerate(zip(indices, etiquetas), 1):
    if etiq == 'peor' and rank == 5:
        print('\n=== PEORES 4 ===')
    r, g, p, cd = muestras[idx]
    titulo = f'[{etiq}] muestra {idx} — CD={cd:.5f}'
    ruta = SALIDA / f'figura_{rank:02d}_{etiq}_{idx}.png'
    guardar_figura(r, g, p, titulo, ruta)

# Guardar figuras en Drive
import shutil, time
ts = time.strftime('%Y%m%d_%H%M')
dst_drive = f'{RUTA_SALIDA_DRIVE}/resultados_v2_{ts}'
shutil.copytree(str(SALIDA), dst_drive, dirs_exist_ok=True)
print(f'\nFiguras guardadas en Drive: {dst_drive}')